# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² Mental Health Survey Dataset (Kilifi County, Kenya) using the `mlcroissant` library. The dataset includes demographic details and psychological assessment scores (GAD-7, PHQ-9, PSQ) from a mental health survey, provided in a FAIR and AI-ready format.

### Dataset Source
The dataset is described via a Croissant schema available at the following URL, with all data elements (record sets, fields, columns) referenced by their `@id` as per the Croissant specification.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and available records from the Croissant schema URL using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Metadata is available as an object with attributes
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")

## 2. Data Overview
Review available record sets, their fields, and column `@id`s. All references are by their `@id` for traceability and reproducibility.

In [ ]:
# List all record sets, their ids, and associated field ids
record_sets_info = []
for rs in dataset.record_sets():
    print(f"RecordSet: @id={rs.id}, name={rs.name}")
    record_sets_info.append({'@id': rs.id, 'name': rs.name})
    print("  Fields and Columns:")
    for field in rs.fields:
        print(f"    Field: @id={field.id}, name={field.name}, data_type={field.data_type}, column_ids={[c.id for c in field.columns]}")
    print()

# You can manually note ids to use in later parts, or use them programmatically below.

## 3. Data Extraction
Load the records of each record set into a pandas DataFrame. Use the record set and field `@id` values from the overview above for precise referencing and extraction.

In [ ]:
# Get all record set @ids discovered above
record_set_ids = [info['@id'] for info in record_sets_info]
dataframes = {}

# Load all records for each record set
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for RecordSet @id={record_set_id}")
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
        print()

# Preview the head of the main survey record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id and main_record_set_id in dataframes:
    print(f"First five records of main record set (@id={main_record_set_id}):")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's process the main record set (e.g., the survey data). We'll filter for high PHQ-9 depression scores, normalize the GAD-7 anxiety scores, and group by a demographic field such as `gender`. 

**All field references use their `@id` - update the variables below if the dataset structure changes.**

In [ ]:
# Set the @id references for the key fields
# Update these according to cell 5 output
record_set_id = main_record_set_id  # Main survey data's record set

# Example Field @ids (adjust as discovered in cell 5 above):
phq9_field = None
gad7_field = None
gender_field = None
for rs in dataset.record_sets():
    if rs.id == record_set_id:
        for field in rs.fields:
            if 'phq' in field.name.lower():
                phq9_field = field.id
            if 'gad' in field.name.lower():
                gad7_field = field.id
            if 'gender' in field.name.lower():
                gender_field = field.id

print(f"Using @ids - PHQ-9: {phq9_field}, GAD-7: {gad7_field}, gender: {gender_field}")

df = dataframes.get(record_set_id, pd.DataFrame())
if not df.empty and phq9_field and gad7_field and gender_field:
    # Ensure numeric types
    df[phq9_field] = pd.to_numeric(df[phq9_field], errors='coerce')
    df[gad7_field] = pd.to_numeric(df[gad7_field], errors='coerce')

    # Filter for high PHQ-9 scores (e.g. >10 indicate moderate/severe)
    threshold = 10
    filtered_df = df[df[phq9_field] > threshold].copy()
    print(f"Filtered records with PHQ-9 (@id={phq9_field}) > {threshold}:")
    display(filtered_df[[phq9_field, gad7_field, gender_field]].head())

    # Normalize GAD-7 scores
    filtered_df[gad7_field + '_normalized'] = (
        filtered_df[gad7_field] - filtered_df[gad7_field].mean()
    ) / filtered_df[gad7_field].std()
    print(f"Normalized GAD-7 (@id={gad7_field}) for filtered records:")
    display(filtered_df[[gad7_field, gad7_field + '_normalized', gender_field]].head())

    # Group by gender field
    if gender_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(gender_field)[[phq9_field, gad7_field]].mean()
        print(f"Mean PHQ-9 and GAD-7 by {gender_field}:")
        display(grouped_df)
else:
    print("Unable to find required fields or records are empty. Check field @ids and record set structure.")

## 5. Visualization
Visualize the distributions of PHQ-9 (depression) and GAD-7 (anxiety) scores, and their relationship by gender, using matplotlib or seaborn.

**All field references use their `@id`.**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and phq9_field and gad7_field:
    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    sns.histplot(df[phq9_field].dropna(), bins=16, kde=True)
    plt.title(f"PHQ-9 Distribution (@id={phq9_field})")
    plt.xlabel('PHQ-9 Score')

    plt.subplot(1, 2, 2)
    sns.histplot(df[gad7_field].dropna(), bins=16, kde=True, color='orange')
    plt.title(f"GAD-7 Distribution (@id={gad7_field})")
    plt.xlabel('GAD-7 Score')
    plt.tight_layout()
    plt.show()

    # Scatterplot by gender
    if gender_field and gender_field in df.columns:
        plt.figure(figsize=(8, 5))
        sns.scatterplot(
            x=df[gad7_field],
            y=df[phq9_field],
            hue=df[gender_field],
            palette='Set2',
            alpha=0.7
        )
        plt.xlabel(f'GAD-7 (@id={gad7_field})')
        plt.ylabel(f'PHQ-9 (@id={phq9_field})')
        plt.title('Anxiety vs Depression Scores by Gender')
        plt.legend(title=gender_field)
        plt.show()
else:
    print("Visualization skipped: required fields or data missing.")

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and process a FAIR²-compliant mental health survey dataset using the `mlcroissant` library. 

- All dataset entities were referenced by their unique `@id`, ensuring reproducibility and traceable analysis.
- We loaded all record sets, explored their fields, and conducted exploratory analysis: filtering respondents with high depression scores (PHQ-9 > 10), normalizing anxiety scores (GAD-7), and comparing means by gender.
- Visualizations highlighted the distributions and interrelations of psychological scores within the dataset.

This approach is recommended for all Croissant-compliant datasets to ensure transparent and reproducible FAIR data science workflows.